# Full Transformer From Scratch: English to French

Build both halves of the original [Transformer](https://arxiv.org/abs/1706.03762): an **encoder** that reads an English sentence and a **decoder** that writes its French translation one token at a time.

```text
English tokens -> embeddings + positions -> Encoder stack -> source memory
                                                                  |
French prefix  -> embeddings + positions -> Decoder stack <-------+ (cross-attention)
                                                  |
                                                  v
                                      Linear -> next French token scores
```

We write the attention math and both stacks. PyTorch provides basic layers, automatic differentiation, and the optimizer. We train fresh weights and build our own vocabularies; no prebuilt Transformer or pretrained model is used.

## 01 - Setup

Run the notebook from top to bottom. The first cell installs PyTorch only if needed. The default run is small enough for a CPU; a Colab GPU is optional. Set `FAST_MODE = False` to train on more sentence pairs for longer.

In [ ]:
%pip install -q "torch>=2.1"

In [ ]:
import math
import random
import re
from collections import Counter
from copy import deepcopy
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
FAST_MODE = True
TRAIN_SIZE = 6_000 if FAST_MODE else 24_000
VAL_SIZE = TEST_SIZE = 500 if FAST_MODE else 1_500
EPOCHS = 8 if FAST_MODE else 16
MAX_LENGTH = 12  # Includes <bos> and <eos>.
MIN_WORD_FREQUENCY = 2
BATCH_SIZE = 64
LEARNING_RATE = 0.001

D_MODEL = 64
NUM_HEADS = 4
NUM_LAYERS = 2  # In each stack.
FF_DIM = 128
DROPOUT = 0.1

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
print("Device:", device)

## 02 - Download and split the sentence pairs

We use English-French pairs from [Tatoeba](https://tatoeba.org/) via [ManyThings](https://www.manythings.org/anki/), downloaded from the [PyTorch translation tutorial](https://docs.pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html). The dataset is not bundled with the notebook.

The tokenizer lowercases text and treats punctuation as separate tokens. We keep short pairs and take one French reference per tokenized English sentence so the same source cannot appear in multiple splits. We shuffle before splitting; the held-out test set is for the final check.

In [ ]:
DATA_URL = "https://download.pytorch.org/tutorial/data.zip"


def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())


def load_pairs():
    with urlopen(DATA_URL, timeout=60) as response:
        data = BytesIO(response.read())

    pairs_by_source = {}
    with ZipFile(data) as archive, archive.open("data/eng-fra.txt") as lines:
        for line in lines:
            english, french = line.decode("utf-8").rstrip("\r\n").split("\t")
            source, target = tokenize(english), tokenize(french)
            if source and target and max(len(source), len(target)) <= MAX_LENGTH - 2:
                pairs_by_source.setdefault(tuple(source), target)
    return list(pairs_by_source.items())


pairs = load_pairs()
random.Random(SEED).shuffle(pairs)
validation_end = TRAIN_SIZE + VAL_SIZE
test_end = validation_end + TEST_SIZE
if len(pairs) < test_end:
    raise ValueError(f"Need {test_end:,} unique short pairs, but found {len(pairs):,}.")
train_pairs = pairs[:TRAIN_SIZE]
validation_pairs = pairs[TRAIN_SIZE:validation_end]
test_pairs = pairs[validation_end:test_end]
print(f"Train: {len(train_pairs):,} | Validation: {len(validation_pairs):,} | Test: {len(test_pairs):,}")

## 03 - Build the vocabularies and batches

English and French have separate vocabularies built **only from training pairs**. `<bos>` marks the start, `<eos>` the end, `<pad>` fills empty slots, and `<unk>` stands for an unknown word.

During training, the decoder gets the correct French prefix and predicts the token **one position to the right**. This is called teacher forcing. For example:

```text
decoder input: <bos>   bonjour   !       <eos>
next token:    bonjour !         <eos>   <pad>   (ignored in the loss)
```

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD, BOS, EOS, UNK = range(len(SPECIAL_TOKENS))


def build_vocabulary(sentences):
    counts = Counter(token for sentence in sentences for token in sentence)
    words = sorted(word for word, count in counts.items() if count >= MIN_WORD_FREQUENCY)
    return {word: index for index, word in enumerate(SPECIAL_TOKENS + words)}


def encode_tokens(tokens, vocabulary):
    content = [vocabulary.get(token, UNK) for token in tokens[: MAX_LENGTH - 2]]
    ids = [BOS] + content + [EOS]
    return ids + [PAD] * (MAX_LENGTH - len(ids))


def make_loader(pairs, shuffle=False):
    source_ids = torch.tensor([encode_tokens(source, source_vocab) for source, _ in pairs])
    target_ids = torch.tensor([encode_tokens(target, target_vocab) for _, target in pairs])
    return DataLoader(
        TensorDataset(source_ids, target_ids),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(SEED),
    )


source_vocab = build_vocabulary(source for source, _ in train_pairs)
target_vocab = build_vocabulary(target for _, target in train_pairs)
target_words = {index: word for word, index in target_vocab.items()}
train_loader = make_loader(train_pairs, shuffle=True)
validation_loader = make_loader(validation_pairs)
print(f"Vocabulary sizes: {len(source_vocab):,} English | {len(target_vocab):,} French")

## 04 - Positions and multi-head attention

Sinusoidal position vectors let the model distinguish word order. Attention turns hidden states into learned **queries** (what to look for), **keys** (what to match), and **values** (what to read). Each head computes `softmax(Q K^T / sqrt(head_dim)) V`. We apply the mask before softmax so disallowed keys get zero weight, then concatenate the heads.

If `B` is the batch size and `H` the number of heads, scores have shape `[B, H, query_length, key_length]`. The two lengths can differ: one French token can look at the whole English sentence.

In [ ]:
class TokenPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_length, dropout):
        super().__init__()
        self.tokens = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.scale = math.sqrt(d_model)
        self.dropout = nn.Dropout(dropout)
        # Keep initial token and position features on a similar scale.
        with torch.no_grad():
            self.tokens.weight.div_(self.scale)

        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        channels = torch.arange(0, d_model, 2, dtype=torch.float32)
        angles = positions / (10_000 ** (channels / d_model))
        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = angles.sin()
        encoding[:, 1::2] = angles[:, : d_model // 2].cos()
        self.register_buffer("positions", encoding)

    def forward(self, token_ids):
        x = self.tokens(token_ids) * self.scale
        return self.dropout(x + self.positions[: token_ids.size(1)])


def scaled_dot_product_attention(queries, keys, values, allowed):
    scores = (queries @ keys.transpose(-2, -1)) / math.sqrt(queries.size(-1))
    scores = scores.masked_fill(~allowed, float("-inf"))
    # An entirely masked row should contribute zero, not softmax(-inf) = NaN.
    scores = scores.masked_fill(~allowed.any(dim=-1, keepdim=True), 0.0)
    weights = scores.softmax(dim=-1).masked_fill(~allowed, 0.0)
    return weights @ values


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if num_heads < 1 or d_model % num_heads != 0:
            raise ValueError("D_MODEL must be divisible by a positive NUM_HEADS.")
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_projection = nn.Linear(d_model, d_model)
        self.key_projection = nn.Linear(d_model, d_model)
        self.value_projection = nn.Linear(d_model, d_model)
        self.output_projection = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, length, _ = x.shape
        # [B, length, D] -> [B, H, length, head_dim]
        return x.reshape(batch_size, length, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, query, key, value, allowed):
        queries = self.split_heads(self.query_projection(query))
        keys = self.split_heads(self.key_projection(key))
        values = self.split_heads(self.value_projection(value))
        heads = scaled_dot_product_attention(queries, keys, values, allowed)
        batch_size, query_length, d_model = query.shape
        combined = heads.transpose(1, 2).reshape(batch_size, query_length, d_model)
        return self.output_projection(combined)

## 05 - The encoder and decoder

| Attention | Query comes from | Keys and values come from | What is hidden? |
|---|---|---|---|
| Encoder self-attention | English | English | English padding |
| Decoder self-attention | French prefix | French prefix | Future positions and French padding |
| Decoder cross-attention | French prefix | Encoder memory | English padding |

Every attention or feed-forward sublayer uses **Add & Norm**: add its output to its input, then apply layer normalization. The feed-forward layers process one position at a time. The final linear layer maps the decoder's hidden states to French vocabulary logits, `[batch, target_length, target_vocab_size]`.

In [ ]:
def feed_forward(d_model, ff_dim):
    return nn.Sequential(nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.ff = feed_forward(d_model, ff_dim)
        self.self_norm = nn.LayerNorm(d_model)
        self.ff_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, source_allowed):
        attended = self.self_attention(x, x, x, source_allowed)
        x = self.self_norm(x + self.dropout(attended))
        return self.ff_norm(x + self.dropout(self.ff(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.ff = feed_forward(d_model, ff_dim)
        self.self_norm = nn.LayerNorm(d_model)
        self.cross_norm = nn.LayerNorm(d_model)
        self.ff_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, target_allowed, source_allowed):
        attended = self.self_attention(x, x, x, target_allowed)
        x = self.self_norm(x + self.dropout(attended))
        attended = self.cross_attention(x, memory, memory, source_allowed)
        x = self.cross_norm(x + self.dropout(attended))
        return self.ff_norm(x + self.dropout(self.ff(x)))

In [ ]:
class Transformer(nn.Module):
    def __init__(
        self, source_vocab_size, target_vocab_size, d_model=D_MODEL, num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS, ff_dim=FF_DIM, dropout=DROPOUT, max_length=MAX_LENGTH,
    ):
        super().__init__()
        if num_layers < 1:
            raise ValueError("The encoder and decoder must each contain at least one layer.")
        self.source_embedding = TokenPositionEmbedding(source_vocab_size, d_model, max_length, dropout)
        self.target_embedding = TokenPositionEmbedding(target_vocab_size, d_model, max_length, dropout)
        self.encoder_layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)
        )
        self.decoder_layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)
        )
        self.to_logits = nn.Linear(d_model, target_vocab_size)

    def encode(self, source_ids):
        # [B, 1, 1, source_length] broadcasts over heads and query positions.
        source_allowed = (source_ids != PAD)[:, None, None, :]
        memory = self.source_embedding(source_ids)
        for layer in self.encoder_layers:
            memory = layer(memory, source_allowed)
        return memory, source_allowed

    def decode(self, target_ids, memory, source_allowed):
        length = target_ids.size(1)
        causal = torch.ones(length, length, dtype=torch.bool, device=target_ids.device).tril()
        target_keys = (target_ids != PAD)[:, None, None, :]
        target_allowed = causal & target_keys

        x = self.target_embedding(target_ids)
        for layer in self.decoder_layers:
            x = layer(x, memory, target_allowed, source_allowed)
        return self.to_logits(x)

    def forward(self, source_ids, target_ids):
        memory, source_allowed = self.encode(source_ids)
        return self.decode(target_ids, memory, source_allowed)

## 06 - Check the model before training

Changing a future French token must not change earlier predictions. Changing the English sentence **should** change the output, even when the French prefix stays the same. We also check padding, unequal source and target lengths, and whether the loss sends a gradient back into the encoder.

In [ ]:
def check_transformer():
    queries = keys = torch.zeros(1, 1, 2, 2)
    values = torch.tensor([[[[1.0, 2.0], [9.0, 9.0]]]])
    allowed = torch.tensor([[[[True, False], [False, False]]]])
    expected = torch.tensor([[[[1.0, 2.0], [0.0, 0.0]]]])
    torch.testing.assert_close(scaled_dot_product_attention(queries, keys, values, allowed), expected)

    tiny = Transformer(20, 25, d_model=16, num_heads=4, ff_dim=32, dropout=0.0)
    source = torch.tensor([[BOS, 4, 5, EOS, PAD], [BOS, 6, 7, EOS, PAD]])
    prefix = torch.tensor([[BOS, 8, 9], [BOS, 8, 9]])
    tiny.eval()
    with torch.no_grad():
        logits = tiny(source, prefix)
        assert logits.shape == (2, 3, 25)

        changed_future = prefix.clone()
        changed_future[:, -1] = 10
        torch.testing.assert_close(tiny(source, changed_future)[:, :-1], logits[:, :-1])
        torch.testing.assert_close(tiny(source[:, :-1], prefix), logits, atol=1e-5, rtol=1e-5)
        assert (logits[0] - logits[1]).abs().max() > 1e-4, "Output must depend on the source."

        padded_prefix = torch.tensor([[BOS, PAD, 9], [BOS, PAD, 9]])
        before = tiny(source, padded_prefix)
        tiny.target_embedding.tokens.weight[PAD].fill_(5.0)
        after = tiny(source, padded_prefix)
        torch.testing.assert_close(before[:, -1], after[:, -1])

    tiny.train()
    labels = torch.tensor([[8, 9, EOS], [8, 9, EOS]])
    loss = nn.CrossEntropyLoss()(tiny(source, prefix).reshape(-1, 25), labels.flatten())
    loss.backward()
    gradient = tiny.encoder_layers[0].self_attention.query_projection.weight.grad
    assert gradient is not None and torch.isfinite(gradient).all() and gradient.abs().max() > 0
    print("Passed: attention masks, output shape, causal decoding, source dependence, and gradients.")


check_transformer()

## 07 - Train with teacher forcing

The causal mask lets us train on all target positions at once without leaking later tokens. Cross-entropy compares each prediction with the next French token and ignores `<pad>`. The epoch loss is averaged over real target tokens (including `<eos>`). We keep the weights with the lowest validation loss.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_tokens = 0.0, 0

    with torch.set_grad_enabled(training):
        for source, target in loader:
            source, target = source.to(device), target.to(device)
            decoder_input = target[:, :-1]
            labels = target[:, 1:]
            token_count = (labels != PAD).sum()

            if training:
                optimizer.zero_grad()
            logits = model(source, decoder_input)
            loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
            if training:
                (loss / token_count).backward()
                optimizer.step()

            total_loss += loss.item()
            total_tokens += token_count.item()
    return total_loss / total_tokens


torch.manual_seed(SEED)
model = Transformer(len(source_vocab), len(target_vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
best_loss, best_weights = float("inf"), None

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer)
    validation_loss = run_epoch(model, validation_loader)
    if not (math.isfinite(train_loss) and math.isfinite(validation_loss)):
        raise RuntimeError(f"Non-finite loss at epoch {epoch}.")
    print(f"Epoch {epoch:2d}/{EPOCHS} | train {train_loss:.3f} | validation {validation_loss:.3f}")
    if validation_loss < best_loss:
        best_loss = validation_loss
        best_weights = deepcopy(model.state_dict())

model.load_state_dict(best_weights);

## 08 - Evaluate and translate

The test loss still uses correct French prefixes. Real generation is different: encode the English sentence once, start the decoder with `<bos>`, and append its most likely next token at each step. Stop at `<eos>` or the length limit. The examples below show the actual generated tokens beside one human reference.

This small model demonstrates the full pipeline, not high-quality translation. It can miss words, generate `<unk>`, or get grammar wrong. The displayed text keeps spaces between tokens, including punctuation.

In [ ]:
@torch.no_grad()
def translate(text):
    tokens = tokenize(text)
    if not tokens:
        raise ValueError("Enter an English sentence to translate.")

    model.eval()
    source = torch.tensor([encode_tokens(tokens, source_vocab)], device=device)
    memory, source_allowed = model.encode(source)
    prefix = torch.tensor([[BOS]], device=device)
    generated = []

    for _ in range(MAX_LENGTH - 1):
        next_logits = model.decode(prefix, memory, source_allowed)[0, -1]
        next_logits[[PAD, BOS]] = float("-inf")
        next_id = next_logits.argmax().item()
        if next_id == EOS:
            break
        generated.append(target_words[next_id])
        prefix = torch.cat([prefix, torch.tensor([[next_id]], device=device)], dim=1)
    return " ".join(generated)


print(f"Test loss: {run_epoch(model, make_loader(test_pairs)):.3f}\n")
for english, french in test_pairs[:5]:
    source_text = " ".join(english)
    print("English:  ", source_text)
    print("Reference:", " ".join(french))
    print("Generated:", translate(source_text), "\n")

In [ ]:
# Edit this sentence and rerun just this cell after training.
sentence = "Where are you?"
print("English:", sentence)
print("French: ", translate(sentence))